# 05 Text Processing

This notebook builds the temporal clinical-note pipeline for multimodal early sepsis detection. It filters notes by category, aligns them to ICU stays, censors them at each horizon-specific prediction time, and creates note-window text artifacts for later transformer embedding.

## Design choices

- Only notes written before prediction time are used.
- Physician, nursing, and radiology notes are retained by default.
- Note text is lightly cleaned, not aggressively normalized, to preserve clinical phrasing.
- Outputs are saved as note-level and window-level text tables.
- Actual transformer embedding is deferred to the multimodal modeling stage, but the preferred model is tracked in config.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [3]:
import pandas as pd

from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths
from src.data_processing.text_processing import (
    aggregate_notes_by_window,
    build_horizon_note_rows,
    load_and_filter_notes,
    summarize_note_coverage,
)

In [4]:
config['text_processing']

{'note_categories': ['Physician', 'Nursing', 'Radiology'],
 'max_notes_per_stay': 200,
 'max_note_characters': 4000,
 'min_note_characters': 20,
 'aggregation_window_hours': 6,
 'pretrained_text_model_name': 'emilyalsentzer/Bio_ClinicalBERT',
 'embedding_backend': 'auto',
 'tokenizer_max_length': 128,
 'embedding_batch_size': 8,
 'local_files_only': False,
 'note_columns': ['ROW_ID',
  'SUBJECT_ID',
  'HADM_ID',
  'CHARTDATE',
  'CHARTTIME',
  'STORETIME',
  'CATEGORY',
  'DESCRIPTION',
  'TEXT']}

## Load structured artifacts from previous stages

In [5]:
cohort_path = paths['processed_data_dir'] / '03_cohort_construction' / 'cohort.csv'
assert cohort_path.exists(), 'Cohort file missing. Run 03_cohort_construction first.'
cohort_df = pd.read_csv(cohort_path, parse_dates=['INTIME', 'OUTTIME', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'DOB', 'DOD'])

horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    name = f'horizon_{int(horizon)}h'
    table_path = paths['processed_data_dir'] / '04_feature_engineering' / f'{name}.csv'
    assert table_path.exists(), f'Missing structured horizon dataset: {name}. Run 04_feature_engineering first.'
    horizon_tables[name] = pd.read_csv(table_path, parse_dates=['hour', 'INTIME', 'OUTTIME', 'prediction_time'], low_memory=False)

{key: value.shape for key, value in horizon_tables.items()}

{'horizon_6h': (1268294, 90),
 'horizon_12h': (1131670, 90),
 'horizon_24h': (860480, 90)}

## Load and filter raw notes

In [6]:
notes_df = load_and_filter_notes(
    extracted_dir=paths['extracted_data_dir'],
    cohort=cohort_df,
    note_categories=config['text_processing']['note_categories'],
    note_columns=config['text_processing']['note_columns'],
    min_note_characters=config['text_processing']['min_note_characters'],
    max_note_characters=config['text_processing']['max_note_characters'],
    max_notes_per_stay=config['text_processing']['max_notes_per_stay'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
)
notes_df.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,note_time,clean_text,text_length,ICUSTAY_ID,INTIME,OUTTIME,note_rank_within_stay
0,768818,3,145834.0,2101-10-20,2101-10-20 22:23:00,NaN,Radiology,CHEST (PORTABLE AP),2101-10-20 22:23:00,[**2101-10-20**] 10:23 PM CHEST (PORTABLE AP) ...,1333,211552.0,2101-10-20 19:10:11,2101-10-26 20:43:09,1
1,768829,3,145834.0,2101-10-21,2101-10-21 01:00:00,NaN,Radiology,CHEST (PORTABLE AP),2101-10-21 01:00:00,[**2101-10-21**] 1:00 AM CHEST (PORTABLE AP) C...,1248,211552.0,2101-10-20 19:10:11,2101-10-26 20:43:09,2
2,768834,3,145834.0,2101-10-21,2101-10-21 06:10:00,NaN,Radiology,CHEST (PORTABLE AP),2101-10-21 06:10:00,[**2101-10-21**] 6:10 AM CHEST (PORTABLE AP) C...,999,211552.0,2101-10-20 19:10:11,2101-10-26 20:43:09,3
3,768885,3,145834.0,2101-10-21,2101-10-21 16:43:00,NaN,Radiology,CHEST (PORTABLE AP),2101-10-21 16:43:00,[**2101-10-21**] 4:43 PM CHEST (PORTABLE AP) C...,1130,211552.0,2101-10-20 19:10:11,2101-10-26 20:43:09,4
4,768950,3,145834.0,2101-10-22,2101-10-22 16:27:00,NaN,Radiology,CHEST (PORTABLE AP),2101-10-22 16:27:00,[**2101-10-22**] 4:27 PM CHEST (PORTABLE AP) C...,801,211552.0,2101-10-20 19:10:11,2101-10-26 20:43:09,5


In [7]:
notes_df[['ICUSTAY_ID', 'CATEGORY']].groupby('CATEGORY').size().sort_values(ascending=False).head(10)

CATEGORY
Nursing      214339
Radiology    208944
dtype: int64

## Build horizon-aware note datasets

In [8]:
note_level_datasets = {}
window_level_datasets = {}
for dataset_name, structured_df in horizon_tables.items():
    note_rows = build_horizon_note_rows(
        notes=notes_df,
        horizon_structured_rows=structured_df,
        aggregation_window_hours=config['text_processing']['aggregation_window_hours'],
    )
    note_level_datasets[f'{dataset_name}_notes'] = note_rows
    window_level_datasets[f'{dataset_name}_note_windows'] = aggregate_notes_by_window(note_rows)

coverage_df = summarize_note_coverage(note_level_datasets)
coverage_df

,dataset_name,note_row_count,icu_stay_count,positive_stay_count,mean_notes_per_stay
0,horizon_6h_notes,213962,27315,1364,7.833132
1,horizon_12h_notes,195090,25254,1025,7.725113
2,horizon_24h_notes,163584,19512,728,8.383764


In [9]:
window_level_datasets['horizon_6h_note_windows'].head() if 'horizon_6h_note_windows' in window_level_datasets else None

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,prediction_time,note_window_index,split,sepsis3_label,note_count,first_note_time,last_note_time,categories,aggregated_text
0,3,145834.0,211552.0,2101-10-26 14:43:09,0,test,0,1,2101-10-26 11:34:00,2101-10-26 11:34:00,Radiology,[**2101-10-26**] 11:34 AM VIDEO OROPHARYNGEAL ...
1,3,145834.0,211552.0,2101-10-26 14:43:09,1,test,0,1,2101-10-26 06:01:00,2101-10-26 06:01:00,Radiology,[**2101-10-26**] 6:01 AM CHEST (PORTABLE AP) C...
2,3,145834.0,211552.0,2101-10-26 14:43:09,5,test,0,1,2101-10-25 06:32:00,2101-10-25 06:32:00,Radiology,[**2101-10-25**] 6:32 AM CHEST (PORTABLE AP) C...
3,3,145834.0,211552.0,2101-10-26 14:43:09,7,test,0,1,2101-10-24 16:06:00,2101-10-24 16:06:00,Radiology,[**2101-10-24**] 4:06 PM CHEST (PORTABLE AP) C...
4,3,145834.0,211552.0,2101-10-26 14:43:09,9,test,0,1,2101-10-24 08:05:00,2101-10-24 08:05:00,Radiology,[**2101-10-24**] 8:05 AM CHEST (PORTABLE AP) C...


## Save reusable text artifacts

In [10]:
output_dir = paths['processed_data_dir'] / '05_text_processing'
artifact_bundle = {
    'filtered_notes': notes_df,
    'note_coverage_summary': coverage_df,
}
artifact_bundle.update(note_level_datasets)
artifact_bundle.update(window_level_datasets)
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'filtered_notes': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/filtered_notes.csv',
 'note_coverage_summary': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/note_coverage_summary.csv',
 'horizon_6h_notes': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/horizon_6h_notes.csv',
 'horizon_12h_notes': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/horizon_12h_notes.csv',
 'horizon_24h_notes': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/horizon_24h_notes.csv',
 'horizon_6h_note_windows': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/horizon_6h_note_windows.csv',
 'horizon_12h_note_windows': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/05_text_processing/horizon_12h_note_windows.csv',
 'horizon_24h_n

In [11]:
manifest_path = paths['manifests_dir'] / '05_text_processing_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='05_text_processing',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'coverage_summary': coverage_df.to_dict(orient='records'),
        'preferred_text_model': config['text_processing']['pretrained_text_model_name'],
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/05_text_processing_manifest.json')

## Review checklist

Before multimodal modeling, verify:

- How many stays have usable notes at each horizon?
- Which note categories dominate coverage?
- Are the aggregated texts short enough for the selected encoder?
- Do positive and negative stays have similar note availability?
- Should additional note categories be included or excluded?

## Next notebook

`06_baseline_models.ipynb` will train structured-data baselines first, giving us strong reference points before the multimodal architectures are introduced.